# Model evaluation matrix: baseline run (Qwen3-1.7B original)

**Settings:** GPU **T4 x2**, Internet **On**. **Inputs:** `bigo-code`, `bigo-test` (rebuild both with `python notebooks/make_bundles.py --only code test`).

Runs `eval.run_all` (Belebele kan/eng/romanized, FLORES chrF++ eng↔kan + Kanglish→eng, speed & ₹ cost, three-route test, agent) — two models in parallel, one per T4. fp16 is chosen automatically on T4. Download `/kaggle/working/results/models/` and copy into the repo's `results/models/`, then run `python -m eval.build_matrix` locally. **Stop the session when done.**

In [ ]:
import os, glob, subprocess
CODE = glob.glob('/kaggle/input/bigo-code*')[0]
TEST = glob.glob('/kaggle/input/bigo-test*')[0]
W = '/kaggle/working'
if not os.path.exists(f'{W}/repo'):
    subprocess.run(['cp', '-r', CODE, f'{W}/repo'], check=True)
os.chdir(f'{W}/repo')
!pip install -q sacrebleu
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3-1.7B')

Quick check first (~10 min): small limits. Then the full run.

In [ ]:
QUICK = True
LIM = ['--limit', '50', '--chrf-limit', '50', '--route-limit', '50'] if QUICK else ['--chrf-limit', '1012']
def run_all(model, tag, gpu):
    return subprocess.Popen(['python', '-m', 'eval.run_all', '--model', model, '--tag', tag, '--data', TEST,
                             '--out', f'{W}/results/models/{tag}', '--device', f'cuda:{gpu}', *LIM],
                            stdout=open(f'{W}/log_{tag}.txt', 'w'), stderr=subprocess.STDOUT)
jobs = [run_all('Qwen/Qwen3-1.7B', 'qwen3_base', 0)]
# optional Indic baseline on GPU 1 (base model, no chat template: chrF/agent are not meaningful for it)
# jobs.append(run_all('sarvamai/sarvam-1', 'sarvam1', 1))
for j in jobs: j.wait()
!tail -5 /kaggle/working/log_*.txt

In [ ]:
!cd /kaggle/working && zip -qr models_out.zip results/models log_*.txt && ls -la models_out.zip

**Later, when the teammate's model arrives** (upload it as a private dataset `bigo-trained`):

```python
M = glob.glob('/kaggle/input/bigo-trained*')[0]
j = run_all(M, 'bigo_trained', 0); j.wait()
# and a standard-BPE-extension model if he trains one: run_all(<path>, 'std_ext_trained', 1)
```

The model folder must contain the `bigo_final` tokenizer files; `model/hf_load.py` refuses to run if AutoTokenizer disagrees with tokenizer.json.